<a href="https://colab.research.google.com/github/krishshah8000/mlproject/blob/master/TF_IDF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

# Load the dataset with latin-1 encoding
df = pd.read_csv('/content/spam.csv', encoding='latin-1')

# Inspect columns to identify targets for removal
print("Original Columns:", df.columns.tolist())

# Drop unnecessary columns that are often present in this specific dataset
# These usually appear as 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4'
cols_to_drop = [col for col in df.columns if 'Unnamed' in col]
df.drop(columns=cols_to_drop, inplace=True)

# Rename v1 and v2 to label and text
df.rename(columns={'v1': 'label', 'v2': 'text'}, inplace=True)

# Display cleaned data info and head
print("\nCleaned DataFrame Info:")
print(df.info())
print("\nFirst 5 rows:")
print(df.head())

Original Columns: ['v1', 'v2', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4']

Cleaned DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   label   5572 non-null   object
 1   text    5572 non-null   object
dtypes: object(2)
memory usage: 87.2+ KB
None

First 5 rows:
  label                                               text
0   ham  Go until jurong point, crazy.. Available only ...
1   ham                      Ok lar... Joking wif u oni...
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...
3   ham  U dun say so early hor... U c already then say...
4   ham  Nah I don't think he goes to usf, he lives aro...


In [ ]:
X = df['text']
y = df['label']

print(f"Shape of X: {X.shape}")
print(f"Shape of y: {y.shape}")

print("\nFirst 5 entries of X (Text Data):")
print(X.head())

print("\nFirst 5 entries of y (Labels):")
print(y.head())

Shape of X: (5572,)
Shape of y: (5572,)

First 5 entries of X (Text Data):
0    Go until jurong point, crazy.. Available only ...
1                        Ok lar... Joking wif u oni...
2    Free entry in 2 a wkly comp to win FA Cup fina...
3    U dun say so early hor... U c already then say...
4    Nah I don't think he goes to usf, he lives aro...
Name: text, dtype: object

First 5 entries of y (Labels):
0     ham
1     ham
2    spam
3     ham
4     ham
Name: label, dtype: object


In [ ]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
import re

# Download necessary NLTK resources including the missing punkt_tab
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('wordnet')
nltk.download('omw-1.4')

# Initialize Lemmatizer and Stopwords
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    # Convert to lowercase
    text = text.lower()
    # Tokenize
    tokens = word_tokenize(text)
    # Filter: remove non-alphabetic, remove stopwords, and lemmatize
    cleaned_tokens = [lemmatizer.lemmatize(word) for word in tokens if word.isalpha() and word not in stop_words]
    return " ".join(cleaned_tokens)

# Apply preprocessing to X
X_preprocessed = X.apply(preprocess_text)

# Verify the transformations
print("Original Text Sample:")
print(X.head())
print("\nPreprocessed Text Sample:")
print(X_preprocessed.head())

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


Original Text Sample:
0    Go until jurong point, crazy.. Available only ...
1                        Ok lar... Joking wif u oni...
2    Free entry in 2 a wkly comp to win FA Cup fina...
3    U dun say so early hor... U c already then say...
4    Nah I don't think he goes to usf, he lives aro...
Name: text, dtype: object

Preprocessed Text Sample:
0    go jurong point crazy available bugis n great ...
1                              ok lar joking wif u oni
2    free entry wkly comp win fa cup final tkts may...
3                  u dun say early hor u c already say
4                  nah think go usf life around though
Name: text, dtype: object


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Initialize TF-IDF Vectorizer
tfidf_vectorizer = TfidfVectorizer(max_features=200,ngram_range=(1,2))

# Fit and transform the preprocessed text data
X_tfidf = tfidf_vectorizer.fit_transform(X_preprocessed)

# Get the feature names (vocabulary)
vocabulary = tfidf_vectorizer.get_feature_names_out()

print(f"Number of unique tokens: {len(vocabulary)}")
print("Sample of vocabulary:", vocabulary[:20])

# Display the shape of the resulting TF-IDF matrix
print(f"\nShape of TF-IDF feature matrix: {X_tfidf.shape}")

# Show the feature vector for the first document
print("\nTF-IDF vector for the first document (sparse representation):")
print(X_tfidf[0])

Number of unique tokens: 200
Sample of vocabulary: ['already' 'also' 'always' 'amp' 'anything' 'around' 'ask' 'babe' 'back'
 'best' 'bit' 'buy' 'ca' 'call' 'call later' 'cant' 'car' 'care' 'cash'
 'chat']

Shape of TF-IDF feature matrix: (5572, 200)

TF-IDF vector for the first document (sparse representation):
<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 4 stored elements and shape (1, 200)>
  Coords	Values
  (0, 50)	0.4375188295365382
  (0, 57)	0.5448525492652294
  (0, 56)	0.4573054767687088
  (0, 183)	0.5500769712299384


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

# Map labels: ham to 0, spam to 1
y_encoded = y.map({'ham': 0, 'spam': 1})

# Split the data into training and testing sets (80/20 split)
X_train, X_test, y_train, y_test = train_test_split(X_tfidf, y_encoded, test_size=0.2, random_state=42)

# Initialize and train the Multinomial Naive Bayes model
nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)

# Make predictions
y_pred = nb_model.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred, target_names=['ham', 'spam'])

print(f"Accuracy: {accuracy:.4f}")
print("\nClassification Report:")
print(report)

Accuracy: 0.9417

Classification Report:
              precision    recall  f1-score   support

         ham       0.95      0.99      0.97       965
        spam       0.89      0.65      0.75       150

    accuracy                           0.94      1115
   macro avg       0.92      0.82      0.86      1115
weighted avg       0.94      0.94      0.94      1115

